# Kaggle Submission Notebook

This notebook is inference-only. Attach:

- the competition dataset
- a Kaggle dataset created from the `artifacts/` folder produced by `final.ipynb`

Then run all cells to create `/kaggle/working/submission.csv`.


In [ ]:
# Optional only if your Kaggle runtime is missing these packages:
# !pip install -q transformers joblib


In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from scipy.special import softmax
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# =========================
# EDIT THESE TWO PATHS
# =========================
COMPETITION_DATA_DIR = Path("/kaggle/input/map-charting-student-math-misunderstandings")
ARTIFACT_DIR = Path("/kaggle/input/map-qwen3-single-model-artifacts/artifacts")
MODEL_DIR = ARTIFACT_DIR / "merged_model"

TEST_PATH = COMPETITION_DATA_DIR / "test.csv"
OUTPUT_PATH = Path("/kaggle/working/submission.csv")


In [ ]:
# =========================
# LOAD ARTIFACTS
# =========================
with open(ARTIFACT_DIR / "metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

le = joblib.load(ARTIFACT_DIR / "label_encoder.joblib")
classes = list(le.classes_)
family_fillers = metadata["family_fillers"]
correct_by_q = {int(k): v for k, v in metadata["correct_answer_map"].items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
infer_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else (
    torch.float16 if torch.cuda.is_available() else torch.float32
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    torch_dtype=infer_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
)
model.eval()

print("Loaded merged offline model from:", MODEL_DIR)


In [ ]:
# =========================
# PREPROCESS TEST
# =========================
test = pd.read_csv(TEST_PATH)
test["QuestionText"] = test["QuestionText"].fillna("")
test["StudentExplanation"] = test["StudentExplanation"].fillna("")
test["MC_Answer"] = test["MC_Answer"].fillna("")

def add_correctness_features(df: pd.DataFrame, correct_map: dict) -> pd.DataFrame:
    df = df.copy()
    df["correct_answer"] = df["QuestionId"].map(lambda q: correct_map.get(int(q)) if pd.notna(q) else None)
    df["is_correct"] = np.where(
        df["correct_answer"].notna(),
        (df["MC_Answer"] == df["correct_answer"]).astype(int),
        pd.NA,
    )
    df["family_prefix"] = df["is_correct"].map(
        lambda x: None if pd.isna(x) else ("True_" if int(x) == 1 else "False_")
    )
    return df

def format_input(row: pd.Series) -> str:
    if pd.isna(row["is_correct"]):
        correct_text = "Unknown"
    else:
        correct_text = "Yes" if int(row["is_correct"]) == 1 else "No"
    return (
        f"Question: {row['QuestionText']}\n"
        f"Answer: {row['MC_Answer']}\n"
        f"Is Correct Answer: {correct_text}\n"
        f"Student Explanation: {row['StudentExplanation']}"
    )

test = add_correctness_features(test, correct_by_q)
test["text"] = test.apply(format_input, axis=1)
test.head()


In [ ]:
# =========================
# INFERENCE
# =========================
def select_top3(prob_row, classes, family_prefix=None, family_fillers=None):
    ranked = sorted(zip(classes, prob_row), key=lambda x: x[1], reverse=True)
    if family_prefix:
        ranked = [x for x in ranked if x[0].startswith(family_prefix)]
    selected = [label for label, _ in ranked[:3]]
    if family_prefix and family_fillers:
        for filler in family_fillers.get(family_prefix, []):
            if filler not in selected:
                selected.append(filler)
            if len(selected) == 3:
                break
    if len(selected) < 3:
        for filler, _ in sorted(zip(classes, prob_row), key=lambda x: x[1], reverse=True):
            if filler not in selected:
                selected.append(filler)
            if len(selected) == 3:
                break
    return selected[:3]

max_length = int(metadata["max_length"])
all_probs = []
model_device = next(model.parameters()).device

with torch.no_grad():
    for text in test["text"].tolist():
        enc = tokenizer(
            text,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        enc = {k: v.to(model_device) for k, v in enc.items()}
        logits = model(**enc).logits.detach().float().cpu().numpy()[0]
        all_probs.append(softmax(logits))

test_top3 = [
    select_top3(
        prob_row=np.asarray(prob_row),
        classes=classes,
        family_prefix=test.loc[i, "family_prefix"],
        family_fillers=family_fillers,
    )
    for i, prob_row in enumerate(all_probs)
]

print("Example predictions:")
for preds in test_top3[:3]:
    print(preds)


In [ ]:
# =========================
# BUILD COMPETITION SUBMISSION
# =========================
submission = pd.DataFrame({
    "Category:Misconception": [" ".join(preds) for preds in test_top3]
})
submission.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)
submission.head()
